# ⚔️ 02: Batallas, ELO y Análisis del Metajuego
Este notebook analiza el comportamiento de los combates y el impacto de los enfrentamientos elementales en la economía de Brokerdex.

**Preguntas clave a responder:**
1. ¿Qué tipos elementales dominan el metajuego en tasa de victorias (*win-rate*)?
2. ¿Existe correlación directa entre el éxito en combate y la cotización de mercado?
3. ¿Cómo se distribuyen los emparejamientos en la matriz de combate?


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from db import get_engine
engine = get_engine()


## 1. Win-Rate por Criatura y Correlación con el Precio Spot
Consultamos victorias, derrotas y precios actuales.


In [ ]:
query_winrates = """
SELECT 
    name,
    type,
    wins,
    losses,
    (wins + losses) AS total_battles,
    ROUND(100.0 * wins / NULLIF(wins + losses, 0), 1) AS win_rate_pct,
    current_price,
    pricing_reference_price
FROM creature_creature
WHERE (wins + losses) > 0
ORDER BY win_rate_pct DESC;
"""

df_wr = pd.read_sql_query(query_winrates, engine)
df_wr


### Dispersión: Win-Rate vs Precio Spot de Mercado


In [ ]:
fig_scatter = px.scatter(
    df_wr,
    x='win_rate_pct',
    y='current_price',
    size='total_battles',
    color='type',
    hover_name='name',
    text='name',
    title="Relación entre Win-Rate (%) y Precio Spot ($)",
    labels={'win_rate_pct': 'Tasa de Victorias (%)', 'current_price': 'Precio Actual ($)'},
    template="plotly_dark"
)
fig_scatter.update_traces(textposition='top center')
fig_scatter.update_layout(height=600)
fig_scatter.show()


## 2. Matriz de Enfrentamientos por Tipo Elemental (Matchup Heatmap)
Analizamos qué tipo elemental supera a cuál con un mapa de calor.


In [ ]:
query_matchups = """
SELECT 
    c1.type AS type_attacker,
    c2.type AS type_defender,
    COUNT(*) AS total_matches,
    SUM(CASE WHEN b.winner_id = bp1.id THEN 1 ELSE 0 END) AS attacker_wins
FROM creature_battle b
JOIN creature_battleparticipant bp1 ON b.id = bp1.battle_id
JOIN creature_battleparticipant bp2 ON b.id = bp2.battle_id AND bp1.id != bp2.id
JOIN creature_creature c1 ON bp1.creature_id = c1.id
JOIN creature_creature c2 ON bp2.creature_id = c2.id
WHERE b.status = 'finished' AND b.winner_id IS NOT NULL
GROUP BY c1.type, c2.type;
"""

df_matchups = pd.read_sql_query(query_matchups, engine)

if not df_matchups.empty:
    df_matchups['win_rate'] = (df_matchups['attacker_wins'] / df_matchups['total_matches']) * 100
    pivot_table = df_matchups.pivot(index='type_attacker', columns='type_defender', values='win_rate').fillna(50)

    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table, annot=True, fmt=".1f", cmap="vlag", center=50, cbar_kws={'label': 'Win Rate (%)'})
    plt.title("Matriz de Enfrentamientos: Win-Rate (%) Tipo Atacante vs Defensor", fontsize=14)
    plt.xlabel("Tipo Defensor")
    plt.ylabel("Tipo Atacante")
    plt.tight_layout()
    plt.show()
else:
    print("Aún no hay suficientes batallas finalizadas registradas para generar la matriz completa.")
